# Polymarket réagit-il aux qualifications de F1 ?

*Projet Python pour la data science - ENSAE 2A, 2026-2027 - Auteurs : Orane Bertout, Stanislas Hanriot, Colombe De Gélis*

Polymarket est un marché de prédiction : pour chaque Grand Prix, il existe un contrat par pilote qui rapporte 1 $ si ce pilote gagne. Le prix du contrat peut donc se lire comme la probabilité de victoire estimée par le marché.

Les qualifications, la veille de la course, fixent la grille de départ. Sur un circuit où il est difficile de dépasser (Monaco par exemple), partir devant décide presque de la course. 

**Question : le marché réagit-il plus fortement aux qualifications sur les circuits où il est difficile de dépasser ?**

Données utilisées :
- Polymarket : prix minute par minute des 24 Grands Prix de 2025 ;
- OpenF1 : horaires des séances 2025 et dépassements enregistrés pendant les courses 2023-2024;
- Jolpica : calendrier et coordonnées des circuits ;

In [ ]:
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

from src.config import ANNEE, ANNEE_COEFF

from src.collect import charger, collecter_polymarket, collecter_openf1, collecter_jolpica
from src.depassement import coef_depassement
from traitement import nettoyer_marches, nettoyer_prix, table_grands_prix


## 1. Récupération des données

La première étape consiste à charger les données qui seront utilisés plus tard.

In [ ]:
events = charger("polymarket_f1_events.parquet", collecter_polymarket, ANNEE)
marches = charger("polymarket_f1_markets.parquet", collecter_polymarket, ANNEE)
prix = charger("polymarket_f1_prices.parquet", collecter_polymarket, ANNEE)
seances = charger("openf1_sessions.parquet", collecter_openf1, ANNEE)
calendrier = charger("jolpica_calendrier.parquet", collecter_jolpica, ANNEE)
coef = charger("coefficients_depassement.parquet", coef_depassement, ANNEE_COEFF)
courses_dep = charger("depassements_par_course.parquet", coef_depassement, ANNEE_COEFF)

print(len(events), "GP,", len(marches), "marchés,", len(prix), "prix")

## 2. Nettoyage des données

Les noms de pilotes ne sont pas toujours écrits pareil d'un GP à l'autre (« Hadjar », « Isack Hadjar », « George Russel »…) et Polymarket ajoute des marchés de réserve (« Other », « Driver A »…). On ramène chaque nom au nom de famille du pilote et on supprime les marchés de réserve.

In [ ]:
marches = nettoyer_marches(marches)
prix = nettoyer_prix(prix, marches)
marches.groupby("event_id")["pilote"].count().value_counts()

Contrôle de la qualité des prix : valeurs manquantes, doublons, valeurs hors de [0, 1] et nombre de points de prix par GP.

In [ ]:
print("Valeurs manquantes :", prix.isna().sum().sum())
print("Doublons :", prix.duplicated(["event_id", "pilote", "timestamp_utc"]).sum())
print("Prix hors de [0, 1] :", (~prix["price"].between(0, 1)).sum())
prix.groupby("event_id").size().describe()

On rassemble ensuite les sources dans une table avec une ligne par Grand Prix : horaires des qualifications et de la course (OpenF1), circuit (Jolpica) et coefficient de dépassement. Toutes les heures sont en UTC dans les trois sources.

In [ ]:
gp = table_grands_prix(events, marches, seances, calendrier)
assert gp["circuit_id"].notna().all(), "GP sans circuit : vérifier la jointure sur la date"

gp = gp.merge(coef, on="circuit_id", how="left")
print("GP sans coefficient de dépassement :", gp["coef_depassement"].isna().sum())
gp[["title", "circuit_id", "quali_debut", "course_debut", "sprint", "coef_depassement"]]

## 3. Coefficient de dépassement

Pour mesurer la difficulté de dépassement, on compte le nombre de dépassement effectuer durant la course. 
Ces données sont récupérées sur les éditions des années 2023 et 2024 et non de 2025 afin de ne pas mesurer le coefficient sur les mêmes courses que la réaction du marché.

In [ ]:
print("Courses 2023-2024 sans données de dépassements :", courses_dep["depassements"].isna().sum())
print("Circuits par nombre d'éditions utilisées :")
print(coef["nb_courses"].value_counts().sort_index())
coef.sort_values("coef_depassement")

## ?. Tests

In [ ]:
from tests.test_depassement import test_depassement, test_courses, test_coef_depassement

test_depassement()
test_courses()
test_coef_depassement()